# Phase 4: SFT — Fine-Tune Qwen3-4B for Tool Calling

Supervised fine-tuning on the teacher trajectories from Phase 2.
Teaches the model: which tools to call, what arguments to pass, when to stop, and how to write the final snapshot.

**Training approach:**
- Base model: `Qwen3-4B` (4-bit quantized)
- LoRA adapters (r=32, alpha=64)
- Train on assistant turns only (system/user/tool turns masked)
- `max_seq_length=8192` (tool outputs already truncated to ~600 tokens)

## Setup

In [ ]:
%pip install -q unsloth trl datasets

In [ ]:
import json
import random
import sys
from datetime import datetime
from pathlib import Path

import mlflow
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_FUNCTIONS, TOOL_SCHEMAS
from bbb.helpers__data_gen import (
    SYSTEM_PROMPT,
    TICKERS,
    _responses_tool_to_chat,
    make_user_prompt,
    trajectory_token_stats,
)
from bbb.helpers__inference import (
    run_local_agent_loop,
    compute_reward,
)

# Derive valid tool names from the tool registry
VALID_TOOL_NAMES = set(TOOL_FUNCTIONS.keys())

# MLflow experiment — matches winterfest convention
mlflow.set_experiment("/Users/kristof.rabay/qwen3-4b-bbb")

run_name = f"qwen3-4b-sft-bbb-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"
print(run_name)

## Load Model + LoRA

In [ ]:
MAX_SEQ_LENGTH = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print(f"Trainable params: {model.print_trainable_parameters()}")

## Load & Format Training Data

Load the SFT trajectories from Phase 2, apply the chat template, and split into train/eval.

In [ ]:
SFT_PATH = PROJECT_ROOT / "data" / "bbb" / "trajectories_sft.jsonl"

sft_records = []
with open(SFT_PATH) as f:
    for line in f:
        sft_records.append(json.loads(line))

print(f"Loaded {len(sft_records)} SFT trajectories")

# Shuffle and split 85/15
random.seed(42)
random.shuffle(sft_records)
split_idx = int(len(sft_records) * 0.85)
train_records = sft_records[:split_idx]
eval_records = sft_records[split_idx:]

print(f"Train: {len(train_records)} | Eval: {len(eval_records)}")

In [ ]:
def format_for_sft(record: dict) -> dict:
    """
    Apply the tokenizer chat template to a Hermes-format trajectory.
    Returns {"text": formatted_string} for SFTTrainer.
    """
    text = tokenizer.apply_chat_template(
        record["messages"],
        tools=record["tools"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


# Format and create HF datasets
train_formatted = [format_for_sft(r) for r in train_records]
eval_formatted = [format_for_sft(r) for r in eval_records]

# Filter out any that are too long
train_formatted = [r for r in train_formatted if r["text"] is not None]
eval_formatted = [r for r in eval_formatted if r["text"] is not None]

train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(eval_formatted)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Eval dataset: {len(eval_dataset)} examples")

In [ ]:
# Inspect a formatted example
sample = train_formatted[0]["text"]
print(f"Token count: {len(tokenizer.encode(sample))}")
print(f"\nFirst 1500 chars:\n{'='*60}")
print(sample[:1500])
print(f"\n...\n\nLast 500 chars:\n{'='*60}")
print(sample[-500:])

In [ ]:
# Token length distribution
lengths = [len(tokenizer.encode(r["text"])) for r in train_formatted]

print(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")
over_limit = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
print(f"Over {MAX_SEQ_LENGTH}: {over_limit}/{len(lengths)}")
if over_limit:
    print("(These will be clipped during training)")

## Training

SFT with `train_on_responses_only` — only assistant turns contribute to the loss.
System, user, and tool messages are masked (labels=-100).

**Note:** If `train_on_responses_only` raises a ZeroDivisionError with Qwen3
(known issue #2771), see the fallback cell below.

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,  # preserve message boundaries for masking

        max_steps=500,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,

        eval_strategy="steps",
        eval_steps=25,
        save_steps=150,
        save_total_limit=3,

        optim="adamw_8bit",
        weight_decay=0.01,

        metric_for_best_model="eval_loss",
        load_best_model_at_end=True,
        logging_steps=5,

        seed=3407,
        output_dir="outputs",
        report_to="mlflow",
        run_name=run_name,
    ),
)

# Mask non-assistant turns — only train on model outputs
# Qwen3 uses <|im_start|> delimiters
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# Verify masking — check that labels are -100 for non-assistant tokens
sample_batch = trainer.get_train_dataloader()
batch = next(iter(sample_batch))

total_tokens = (batch["labels"] != -100).sum().item()
masked_tokens = (batch["labels"] == -100).sum().item()

print(f"Training tokens: {total_tokens}")
print(f"Masked tokens:   {masked_tokens}")
print(f"Train ratio:     {total_tokens / (total_tokens + masked_tokens) * 100:.1f}%")

In [ ]:
trainer_stats = trainer.train()

## Training Analysis

In [ ]:
import pandas as pd

# Extract training history
df_history = pd.DataFrame(trainer.state.log_history)

train_df = df_history[df_history["loss"].notna()]
eval_df = df_history[df_history["eval_loss"].notna()]

if not eval_df.empty:
    final_train = train_df["loss"].iloc[-5:].mean()
    final_eval = eval_df["eval_loss"].iloc[-3:].mean()
    gap = final_eval - final_train

    print(f"Final train loss (avg last 5): {final_train:.4f}")
    print(f"Final eval loss (avg last 3):  {final_eval:.4f}")
    print(f"Gap: {gap:.4f}")
    if gap > 0.1:
        print("Warning: possible overfitting")
    else:
        print("Good generalization")
else:
    print(f"Final train loss: {train_df['loss'].iloc[-1]:.4f}")

In [ ]:
# Plot loss curves
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_df["step"], train_df["loss"], label="Train", alpha=0.7)
    if not eval_df.empty:
        ax.plot(eval_df["step"], eval_df["eval_loss"], label="Eval", marker="o")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("SFT Training — Tool-Calling Qwen3-4B")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — skipping plot")

## Inference Test

Run the fine-tuned model on unseen tickers and compare with baseline.

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

from bbb.tools import TOOL_SCHEMAS

TOOLS_CHAT = [_responses_tool_to_chat(t) for t in TOOL_SCHEMAS]

In [ ]:
# Pick tickers NOT in the training set
train_tickers = {r["ticker"] for r in train_records}
unseen_tickers = [t for t in TICKERS if t not in train_tickers][:10]
print(f"Testing on {len(unseen_tickers)} unseen tickers: {unseen_tickers}")

In [ ]:
sft_results = []

for ticker in unseen_tickers:
    focus = random.choice(["financial health", "growth potential", "competitive position"])
    prompt = make_user_prompt(ticker, focus)

    print(f"\n{ticker} — {focus}")

    try:
        res = run_local_agent_loop(
            model=model,
            tokenizer=tokenizer,
            user_prompt=prompt,
            system_prompt=SYSTEM_PROMPT,
            tools_chat=TOOLS_CHAT,
            tool_functions=TOOL_FUNCTIONS,
            max_iterations=8,
            enable_thinking=True,
            verbose=True,
        )

        reward = compute_reward(res["messages"], VALID_TOOL_NAMES)
        sft_results.append({
            "ticker": ticker,
            "n_tool_calls": res["n_tool_calls"],
            "output_len": len(res["output_text"]),
            "reward": reward,
        })
        print(f"  → reward={reward['total']}, tools={res['n_tool_calls']}, output={len(res['output_text'])} chars")

    except Exception as e:
        print(f"  → ERROR: {e}")
        sft_results.append({"ticker": ticker, "reward": {"total": -3.0}})

# Summary
sft_rewards = [r["reward"]["total"] for r in sft_results]
print(f"\n=== SFT MODEL ===")
print(f"Avg reward: {sum(sft_rewards)/len(sft_rewards):.2f}")
print(f"Completion rate: {sum(1 for r in sft_results if r['reward'].get('completion')) / len(sft_results) * 100:.0f}%")

In [ ]:
# Compare with baseline (load if available)
baseline_path = PROJECT_ROOT / "data" / "bbb" / "baseline_results.jsonl"
if baseline_path.exists():
    baseline_rewards = []
    with open(baseline_path) as f:
        for line in f:
            r = json.loads(line)
            baseline_rewards.append(r["reward"]["total"])

    print(f"{'Metric':<20} {'Baseline':>10} {'SFT':>10}")
    print("-" * 42)
    print(f"{'Avg Reward':<20} {sum(baseline_rewards)/len(baseline_rewards):>+10.2f} {sum(sft_rewards)/len(sft_rewards):>+10.2f}")
else:
    print("No baseline results found — run Phase 3 first for comparison")

## Save Model

Save LoRA adapters for later use (RL, inference, deployment).

In [ ]:
save_dir = PROJECT_ROOT / "models" / "qwen3-4b-tool-calling-sft"
save_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

print(f"Saved LoRA adapters to {save_dir}")

In [ ]:
# Optional: save as GGUF for llama.cpp inference
# Uncomment to export (takes a few minutes)

# gguf_dir = PROJECT_ROOT / "models" / "qwen3-4b-tool-calling-sft-gguf"
# model.save_pretrained_gguf(
#     str(gguf_dir),
#     tokenizer,
#     quantization_method="Q8_0",
# )
# print(f"Saved GGUF to {gguf_dir}")